# Iron Powder - Direct TOF Analysis

This notebook analyzes **real iron powder time-of-flight data**.

**Important**: This data is from a **direct TOF measurement** (pulsed source), not a chopper-modulated source. Therefore, we calculate transmission directly without FOBI reconstruction.

## Data Specifications

- **Sample**: Iron powder (BCC structure)
- **Flight path (L)**: 9 meters  
- **Time resolution**: 10 µs per stack
- **Measurement type**: Direct time-of-flight (not chopper-modulated)
- **Expected Bragg edges**:
  - Fe (110): 2.027 Å
  - Fe (200): 2.866 Å
  - Fe (211): 4.050 Å

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Load Data

In [ ]:
# Load raw data
signal_df = pd.read_csv('iron_powder.csv')
openbeam_df = pd.read_csv('openbeam.csv')

print("Data structure:")
print(signal_df.head())
print(f"\nTotal stacks: {len(signal_df)}")

## 2. Calculate Wavelength

For direct TOF data:
$$\lambda (Å) = 3.956 \times \frac{t (ms)}{L (m)}$$

In [ ]:
# Parameters
time_step = 10  # µs per stack
L = 9  # meters

# Calculate time and wavelength
stack = signal_df['stack'].values
time = stack * time_step  # µs
wavelength = 3.956 * (time / 1000) / L  # Å

print(f"Time range: [{time.min():.0f}, {time.max():.0f}] µs")
print(f"Wavelength range: [{wavelength.min():.3f}, {wavelength.max():.3f}] Å")

## 3. Calculate Direct Transmission

For direct TOF (no chopper):
$$T(\lambda) = \frac{I_{sample}(\lambda)}{I_{openbeam}(\lambda)}$$

In [ ]:
# Extract counts
signal_counts = signal_df['counts'].values
openbeam_counts = openbeam_df['counts'].values

# Calculate transmission
transmission = signal_counts / openbeam_counts

print(f"Transmission range: [{transmission.min():.3f}, {transmission.max():.3f}]")
print(f"Mean transmission: {transmission.mean():.3f}")

## 4. Full Spectrum View

In [ ]:
plt.figure(figsize=(14, 8))

plt.subplot(3, 1, 1)
plt.plot(wavelength, signal_counts, 'b-', linewidth=1, alpha=0.7)
plt.ylabel('Sample (counts)', fontsize=11)
plt.title('Iron Powder: Direct TOF Measurement', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 2)
plt.plot(wavelength, openbeam_counts, 'orange', linewidth=1, alpha=0.7)
plt.ylabel('Open Beam (counts)', fontsize=11)
plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 3)
plt.plot(wavelength, transmission, 'g-', linewidth=1.5, alpha=0.8)
plt.ylabel('Transmission', fontsize=11)
plt.xlabel('Wavelength (Å)', fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Zoom on Bragg Edge Region (1-6 Å)

In [ ]:
# Filter to Bragg edge region
mask = (wavelength >= 1) & (wavelength <= 6)

plt.figure(figsize=(14, 6))
plt.plot(wavelength[mask], transmission[mask], 'b-', linewidth=2)

# Mark expected Bragg edges
edges = [(2.027, 'Fe (110)'), (2.866, 'Fe (200)'), (4.050, 'Fe (211)')]
for edge_pos, edge_label in edges:
    plt.axvline(edge_pos, color='red', linestyle='--', alpha=0.5, linewidth=2)
    plt.text(edge_pos, plt.ylim()[1] * 0.95, f'{edge_label}\n{edge_pos:.3f} Å',
             ha='center', fontsize=10, color='red', fontweight='bold')

plt.xlabel('Wavelength (Å)', fontsize=12)
plt.ylabel('Transmission', fontsize=12)
plt.title('Iron Powder Transmission Spectrum - Bragg Edge Region', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Individual Edge Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

window = 0.5  # Å on each side

for ax, (edge_pos, edge_label) in zip(axes, edges):
    mask_edge = (wavelength >= edge_pos - window) & (wavelength <= edge_pos + window)
    
    ax.plot(wavelength[mask_edge], transmission[mask_edge], 'b-', linewidth=2)
    ax.axvline(edge_pos, color='red', linestyle='--', alpha=0.5, linewidth=2)
    ax.set_xlabel('Wavelength (Å)', fontsize=11)
    ax.set_ylabel('Transmission', fontsize=11)
    ax.set_title(f'{edge_label}\n{edge_pos:.3f} Å', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Apply Simple Smoothing (Optional)

In [ ]:
from scipy.signal import savgol_filter

# Apply Savitzky-Golay filter for smoothing
window_length = 11  # Must be odd
poly_order = 3

transmission_smooth = savgol_filter(transmission, window_length, poly_order)

# Plot comparison
plt.figure(figsize=(14, 6))

mask = (wavelength >= 1.5) & (wavelength <= 5)
plt.plot(wavelength[mask], transmission[mask], 'b-', alpha=0.4, linewidth=1, label='Raw')
plt.plot(wavelength[mask], transmission_smooth[mask], 'r-', linewidth=2, label='Smoothed')

# Mark edges
for edge_pos, edge_label in edges:
    plt.axvline(edge_pos, color='gray', linestyle='--', alpha=0.3)

plt.xlabel('Wavelength (Å)', fontsize=12)
plt.ylabel('Transmission', fontsize=12)
plt.title('Raw vs Smoothed Transmission', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Save Processed Data

In [ ]:
# Create DataFrame with results
results_df = pd.DataFrame({
    'wavelength_angstrom': wavelength,
    'time_us': time,
    'transmission': transmission,
    'transmission_smooth': transmission_smooth,
    'signal_counts': signal_counts,
    'openbeam_counts': openbeam_counts
})

# Save to CSV
results_df.to_csv('iron_powder_analysis.csv', index=False)
print("✅ Results saved to iron_powder_analysis.csv")

# Also save filtered range (1-10 Å)
mask_filtered = (wavelength >= 1) & (wavelength <= 10)
results_filtered = results_df[mask_filtered]
results_filtered.to_csv('iron_powder_1to10A.csv', index=False)
print("✅ Filtered data (1-10 Å) saved to iron_powder_1to10A.csv")

## Summary

This notebook analyzed **direct time-of-flight** neutron transmission data from iron powder.

### Key Findings:
- ✅ Three Bragg edges clearly visible
- ✅ Fe (110) at ~2.027 Å
- ✅ Fe (200) at ~2.866 Å
- ✅ Fe (211) at ~4.050 Å

### Method:
This is **direct TOF analysis** (not FOBI reconstruction):
1. Calculate wavelength from time-of-flight
2. Compute transmission = sample / openbeam
3. Apply optional smoothing

### When to Use FOBI:
FOBI reconstruction is needed for **chopper-modulated** data where:
- Multiple chopper repetitions overlap
- Need to deconvolve chopper response function
- Working with instruments like POLDI, BOA, etc.

For direct TOF data like this example, simple transmission calculation is sufficient!